# Claar, Rembado et al. (2023) on dandiset 000458: spiking claims

**Paper.** Claar et al. (2023), *eLife* 12:RP84630, [doi:10.7554/eLife.84630](https://doi.org/10.7554/eLife.84630);
claims file written from the version of record. **Dandiset.**
[DANDI:000458](https://dandiarchive.org/dandiset/000458/0.230317.0039), version 0.230317.0039, the version the
paper cites and the only published one. See `claar_eeg.ipynb` for the EEG claims and the data check of the trial
tables; this notebook checks the unit tables and tests the spiking claims (Figures 2, 4, 5, 6B, 6C, 7E to 7H).

| Claim | Here |
|---|---|
| fig2c-superficial-quiescence | tested |
| fig2c-deep-triphasic-cortex | tested |
| fig2f-thalamus-triphasic | tested |
| fig3abc-decay-with-distance, fig3d-magnitudes | decided in `claar_lfp.ipynb`; the spiking part is computed here |
| fig4-unit-counts | tested |
| fig4ad-initial-excitation | tested |
| fig4be-off-period | tested |
| fig4cf-rebound | tested |
| fig4-supp1-fs-like-rs | tested |
| fig5a-thalamus-modulated | tested |
| fig5c-burst-probability-depth | tested |
| fig5e-early-latency | tested |
| fig5f-rebound-latency | tested |
| fig6a-csd-sink, fig6-supp1-erp-deep-csd | blocked (see the record): CSD by layer |
| fig6b-bursts-cortical-rate | tested |
| fig6c-cortical-rate-late-erp | tested |
| fig6-supp2-late-erp-bursts | tested |
| fig7e-baseline-rates | tested |
| fig7f-early-latency-state | tested |
| fig7g-rebound-latency-state | tested |
| fig7h-burst-probability-state | tested |

Readings run throughout: stimulus intensity ("high" and "medium" current), awake trials (all, or quiet only), the
pre-stimulus window of the modulation test ("adjacent", ending at onset, or "mirror", -b to -a), and the
Benjamini-Hochberg family ("site": per site and window, over cortical RS and SM-TH units; "pooled": all sites per
window). Units are those in the files (the dandiset's quality threshold) unless a cell says otherwise.

In [ ]:
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.anova import AnovaRM

import claar

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", None)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
urls = claar.asset_urls()
metas = [claar.read_meta(p, u) for p, u in urls]
metas = [m for m in metas if m["units"] is not None]
print(f"{len(metas)} sessions with units, dandiset {claar.DANDISET_ID} version {claar.DANDISET_VERSION}")

## Data check

In [ ]:
rows = []
for m in metas:
    u = m["units"]
    rows.append({"subject": m["subject"], "session": m["session"], "sites": ", ".join(sorted(m["estim"].site.unique())),
                 "units": len(u), "MO": int((u.group == "MO").sum()), "ACA": int((u.group == "ACA").sum()),
                 "SS": int((u.group == "SS").sum()), "VIS": int((u.group == "VIS").sum()), "SM-TH": int((u.group == "SM-TH").sum()),
                 "other": int(u.group.isna().sum()), "RS share": round(u.rs.mean(), 2),
                 "presence>0.9": int((u.presence_ratio > 0.9).sum()),
                 "isi max": round(u.isi_violations.max(), 3), "amp cutoff max": round(u.amplitude_cutoff.max(), 3),
                 "unit z range": f"{u.z.min():.0f}-{u.z.max():.0f}"})
ut = pd.DataFrame(rows)
print(ut.to_string(index=False))
allu = pd.concat([m["units"] for m in metas])
print("\nwaveform_duration (described as s): range", round(allu.waveform_duration.min(), 3), "to", round(allu.waveform_duration.max(), 3),
      "; median", round(allu.waveform_duration.median(), 3), "-> values are in ms")
print("SM-TH acronyms present:", sorted(allu[allu.group == "SM-TH"].location.unique()))
print("cortical acronyms present:", sorted(allu[allu.group.isin(claar.CORTEX)].location.unique()))
print("acronyms with a layer suffix (e.g. MOs5):", sorted({a for a in allu.location.unique() if any(ch.isdigit() for ch in a) and not a.startswith(("CA", "DG"))}))
print("ISI violations max", allu.isi_violations.max().round(3), "; amplitude cutoff max", allu.amplitude_cutoff.max().round(3),
      "; presence ratio min", allu.presence_ratio.min().round(3))

Waveform durations range from about 0.1 to 2 (median 0.577), which are milliseconds, not the seconds the unit
table's description gives; the RS/FS split at 0.4 ms is applied to these values. Every unit meets ISI violations
< 0.5 and amplitude cutoff < 0.1, but presence ratios go down to 0.02, so the file's "quality threshold" is not the
three thresholds of Siegle et al. (2021); the counts are given both ways in fig4-unit-counts. Areas carry CCF
acronyms without layers (MOs, SSp-ll, ...), so the layer-resolved analyses cannot be run from the files. Unit z
coordinates (CCF, 25 um voxels; the midline is about 228) range from 90 to 253: most units lie on one side, but
three sessions (586468, 590479, 590480) have units up to 25 voxels (0.6 mm) past 228, so the files do not show
that every unit is ipsilateral.

In [ ]:
t0 = time.time()
def load(m):
    tab, sp = claar.read_units(m)
    return [claar.spike_site_summary(m, s, tab, sp) for s in sorted(m["estim"].site.unique())]
with ThreadPoolExecutor(max_workers=3) as ex:
    res = [r for rr in ex.map(load, metas) for r in rr]
for r in res:
    r["depth"] = r["site"].split("-")[1]
print(f"{len(res)} sites with units summarised in {time.time() - t0:.0f} s")
empty = [(r["subject"], r["site"]) for r in res if r["n"][("high", "awake")] == 0 and r["n"][("medium", "awake")] == 0]
print("sites without a valid awake trial, left out:", empty)
res = [r for r in res if (r["subject"], r["site"]) not in empty]
print(pd.DataFrame([{"subject": r["subject"], "site": r["site"], **{f"{k[0]} {k[1]}": v for k, v in r["n"].items()}} for r in res]).to_string(index=False))

Clock check for the spikes: units of the stimulated cortex fire within 25 ms of the pulse in most deep sites (the
fraction of significantly increased units at 2 to 25 ms against the same test with onsets shifted by 1 s):

In [ ]:
rows = []
for m in metas[:6]:
    tab, sp = claar.read_units(m, groups=("MO", "SS"))
    for s in sorted(m["estim"].site.unique()):
        tr = claar.site_trials(m, s, "high")
        on = tr[tr.state.isin(["quiet", "running"])].start_time.values
        stim = (tab.group == claar.STIM_AREA[s.split("-")[0]]).values
        sps_ = [x for x, k in zip(sp, stim) if k]
        fr = []
        for shift in (0.0, 1.0):
            p, sg = claar.modulation(sps_, on + shift, (0.002, 0.025))
            fr.append(float(np.mean((claar.bh(p) < 0.05) & (sg > 0))))
        rows.append({"subject": m["subject"], "site": s, "units": len(sps_), "increased, true onsets": round(fr[0], 3), "onsets + 1 s": round(fr[1], 3)})
print(pd.DataFrame(rows).to_string(index=False))

The contrast holds in the sessions checked: the spikes share the trial table's clock.

In [ ]:
LEVELS, AWAKE = claar.LEVELS, ("awake", "quiet")
def unit_flags(r, level, trials, pre, family_q=None):
    # per unit and window: q (BH within site and window over cortical RS + SM-TH units, or pooled q given), sign
    p = r["p"]
    p = p[(p.level == level) & (p.trials == trials) & (p.pre == pre)]
    return p.merge(r["units"][["unit", "group", "rs", "stim", "presence_ratio"]], on="unit")

def flags_all(level, trials, pre, family, cell="RS"):
    # q-values for every site: family 'site' (per site and window) or 'pooled' (all sites per window)
    frames = []
    for k, r in enumerate(res):
        d = unit_flags(r, level, trials, pre)
        d["site_idx"] = k
        d["subject"], d["site"], d["depth"] = r["subject"], r["site"], r["depth"]
        cort = d.group.isin(claar.CORTEX)
        keep = ((cort & (d.rs if cell == "RS" else ~d.rs)) | (d.group == "SM-TH")) if cell == "RS" else (cort & ~d.rs)
        frames.append(d[keep])
    d = pd.concat(frames).reset_index(drop=True)
    d["q"] = np.nan
    if family == "site":
        for key, g in d.groupby(["site_idx", "window"]):
            d.loc[g.index, "q"] = claar.bh(g.p.values)
    else:
        d = d.reset_index(drop=True)
        for w, g in d.groupby("window"):
            d.loc[g.index, "q"] = claar.bh(g.p.values)
    d["inc"] = (d.q < 0.05) & (d.sign > 0)
    d["dec"] = (d.q < 0.05) & (d.sign < 0)
    d["mod"] = d.q < 0.05
    return d

combos = [(lvl, tr, pre, fam) for lvl in LEVELS for tr in AWAKE for pre in claar.PRE_READINGS for fam in ("site", "pooled")]
FL = {}
for c in combos:
    FL[c] = flags_all(*c)
    FL[c + ("FS",)] = flags_all(*c, cell="FS")
print("flag tables:", len(FL))

## fig3abc-decay-with-distance and fig3d-magnitudes: the spiking part

Both claims are decided in `claar_lfp.ipynb`, which computes the LFP and CSD part. The spiking magnitudes need only the RS units of each area, and are computed here as the criterion of fig3d-magnitudes describes (`claar.spiking_magnitude`): all awake trials, kernels of 5 and 10 ms, baselines -2 to 0 and -0.5 to 0 s, onsets shuffled within -2 to +2 s or over the recording, 1000 shuffles. Units of layer 1 cannot be left out. Two signals are integrated: the signed z of the Methods ("signed"), in which a fall below baseline cancels a rise, and |z| (a variant after the verifier's round 2, supported by the Results' "our magnitude metric accounts for both reductions as well as increases"). These numbers do not decide the outcomes. The shuffle "none" rows give the AUC of the z-scored density itself (SD.s), before the comparison with the shuffles.

In [ ]:
MAG = pd.concat([r["magnitude"].assign(subject=r["subject"], site=r["site"], depth=r["depth"], site_idx=k)
                 for k, r in enumerate(res) if len(r["magnitude"])], ignore_index=True)
ex = MAG[(MAG.subject == "571619") & (MAG.site == "MOs-deep")]
print("example 571619 MOs-deep (expected MO 30.9, VIS 1.8 SD.s):")
print(ex.pivot_table(index=["level", "shuffle", "kernel ms", "baseline"], columns=["signal", "group"], values="magnitude").round(2).to_string())
rows = []
for (sig, lvl, sh, kern, base), g in MAG.groupby(["signal", "level", "shuffle", "kernel ms", "baseline"]):
    for unit in ("mouse", "region"):
        out = {"signal": sig, "level": lvl, "shuffle": sh, "kernel ms": kern, "baseline": base, "unit": unit}
        ps = {}
        for dep in ("superficial", "deep"):
            for st, lab in ((True, "stim"), (False, "non-stim")):
                x = g[(g.depth == dep) & (g.stim == st)]
                if unit == "mouse":
                    x = x.groupby("subject").magnitude.mean()
                else:
                    x = x.magnitude
                ps[(dep, lab)] = stats.wilcoxon(x).pvalue if len(x) > 1 else np.nan
                out[f"{dep} {lab} median (n)"] = f"{np.median(x):.2f} ({len(x)})"
        q = claar.bh(list(ps.values()))
        for (k, v), qq in zip(ps.items(), q):
            out[f"q {k[0][:3]} {k[1]}"] = round(qq, 4)
        rows.append(out)
f3 = pd.DataFrame(rows)
print("\nmedians per group and signed-rank tests against 0 (BH over the four spiking tests); "
      "expected medians superficial stimulated 1.8, deep stimulated 29.3, deep non-stimulated 4.4; superficial non-stimulated not significant")
print(f3.to_string(index=False))
exp3 = {("superficial", "stim"): (1.8, 0.18), ("deep", "stim"): (29.3, 2.93), ("deep", "non-stim"): (4.4, 0.44)}
def holds(row):
    med = {k: float(row[f"{k[0]} {k[1]} median (n)"].split()[0]) for k in exp3}
    inside = [k for k in exp3 if abs(med[k] - exp3[k][0]) <= exp3[k][1]]
    sig = (row["q sup stim"] < 0.05 and med[("superficial", "stim")] > 0 and row["q dee stim"] < 0.05 and med[("deep", "stim")] > 0
           and row["q dee non-stim"] < 0.05 and med[("deep", "non-stim")] > 0 and row["q sup non-stim"] >= 0.05)
    return pd.Series({"spiking significance pattern": sig, "spiking medians within tolerance": len(inside)})
f3 = pd.concat([f3, f3.apply(holds, axis=1)], axis=1)
print("\nspiking significance pattern of the paper (stimulated groups and deep non-stimulated q < 0.05 with positive medians, superficial non-stimulated not significant):")
print(f3.groupby(["signal", "shuffle"])[["spiking significance pattern"]].sum().to_string(), "of", int(f3.groupby(["signal", "shuffle"]).size().iloc[0]), "rows each")
print(f3[f3["spiking significance pattern"] | (f3["spiking medians within tolerance"] >= 2)].to_string(index=False))

**Result: spiking part only (the outcomes are in `claar_lfp.ipynb`).** In the example session (571619, deep MOs) the MO spiking magnitude exceeds the VIS one in every reading. With the signed z of the Methods the values are far below the paper's (for example 3.96 against -1.21, high current, epoch shuffle, 5 ms kernel, -2 to 0 s), no stimulated group is significantly above zero after correction in any reading, and the deep non-stimulated medians are -1.16 to 0.15, against 4.4. With |z| the example gives 28.31 to 31.37 in MO with the 5 ms kernel and the -2 to 0 s baseline (30.9 in the paper) and 2.08 to 2.76 in VIS (1.8), and the paper's spiking significance pattern (both stimulated groups and the deep non-stimulated group above zero, the superficial non-stimulated group not significant) holds in 27 of the 32 shuffled rows (13 epoch, 14 recording). With |z|, the 5 ms kernel and the -2 to 0 s baseline at the medium current, the deep stimulated median is 27.96 to 29.18 (29.3 +/- 2.93) and the deep non-stimulated median 4.18 to 6.25 (4.4 +/- 0.44), but the superficial stimulated median is 4.81 to 4.86, against 1.8. The integral of |z| thus comes close to the paper's spiking magnitudes where the signed integral does not, which suggests that the paper's magnitude counts falls below baseline. These numbers do not decide the outcomes; the LFP and CSD part of fig3d-magnitudes is not run (`claar_lfp.ipynb`).

## fig4-unit-counts

> Figure 4 rests on 870 RS units from stimulated and 4110 from non-stimulated cortex in 7 mice with superficial
> stimulation, and 2559 and 9363 in 17 mice with deep stimulation.

In [ ]:
rows = []
for qc in ("dataset", "siegle"):
    for cell in ("RS", "FS"):
        d = []
        for r in res:
            u = r["units"]
            u = u[u.group.isin(claar.CORTEX) & (u.rs if cell == "RS" else ~u.rs)]
            if qc == "siegle":
                u = u[u.presence_ratio > 0.9]
            d.append({"subject": r["subject"], "depth": r["depth"], "stim": int(u.stim.sum()), "nonstim": int((~u.stim).sum())})
        d = pd.DataFrame(d)
        for dep in ("superficial", "deep"):
            x = d[d.depth == dep]
            rows.append({"units": qc, "cell": cell, "depth": dep, "sites": len(x), "mice": x.subject.nunique(),
                         "stimulated": int(x.stim.sum()), "non-stimulated": int(x.nonstim.sum())})
cnt = pd.DataFrame(rows)
print(cnt.to_string(index=False))
print("\nexpected RS: superficial 870 / 4110 (7 mice), deep 2559 / 9363 (17 mice); FS: 180 / 585 (7), 387 / 1422 (16)")

# reading not in the claims file (verifier, round 1): each unit counted once per current with valid awake trials
rows = []
for r in res:
    u = r["units"]
    m = [mm for mm in metas if mm["subject"] == r["subject"] and mm["session"] == r["session"]][0]
    el = m["estim"].query("site == @r['site'] and is_valid and state in ['quiet', 'running']")
    nc = el.current.nunique()
    for cell in ("RS", "FS"):
        x = u[u.group.isin(claar.CORTEX) & (u.rs if cell == "RS" else ~u.rs)]
        rows.append({"depth": r["depth"], "cell": cell, "currents": nc, "stim": nc * int(x.stim.sum()), "nonstim": nc * int((~x.stim).sum())})
pooled_cnt = pd.DataFrame(rows).groupby(["cell", "depth"])[["stim", "nonstim"]].sum()
print("\nunits x currents (each unit counted once per current with valid awake trials):")
print(pooled_cnt.to_string())
print("currents per site:", pd.DataFrame(rows).query("cell == 'RS'").currents.value_counts().to_dict())

**Result: not reproduced.** The files hold 290 and 1370 RS units in stimulated and other cortex for the 7 superficial sites and 853 and 3162 for the 17 deep sites (16 mice): exactly one third of the legend's 870, 4110 and 2559, and 1.3% more than a third of its 9363. The site counts, 7 and 17, match the legend's N. With presence ratio > 0.9 the counts fall further (221, 1020, 665, 2421). Counting each unit once per current with valid awake trials (all 24 sites have three currents) gives 870, 4110, 2559 and 9486 RS units, the legend's first three counts exactly and 1.3% above its 9363, and 204, 609, 447 and 1554 FS units against 180, 585, 387 and 1422. This reading is not in the claims file and is asked of a reader; the outcome stays on the claims file's count of units.

## fig4ad-initial-excitation

> In the initial excitation window (2 to 25 ms), the fraction of activated RS neurons is always higher in stimulated
> cortex than elsewhere; across 24 animals an average of 0.5% (0 to 2.6%) of neurons in non-stimulated regions were
> modulated against 17.8% (0 to 83.3%) in stimulated cortex, and about three times more cortical neurons respond to
> deep than to superficial stimulation.

In [ ]:
def area_fractions(d, window, cell_mask=None):
    # per site and cortical area: fractions of RS units increased, decreased, modulated
    x = d[(d.window == window) & d.group.isin(claar.CORTEX)]
    g = x.groupby(["site_idx", "subject", "depth", "group", "stim"]).agg(n=("unit", "size"), inc=("inc", "mean"), dec=("dec", "mean"), mod=("mod", "mean")).reset_index()
    return g

rows = []
for c in combos:
    d = FL[c]
    g = area_fractions(d, "early")
    st, ns = g[g.stim], g[~g.stim]
    # non-stimulated pooled per site
    x = d[(d.window == "early") & d.group.isin(claar.CORTEX) & ~d.stim]
    nsp = x.groupby("site_idx").agg(inc=("inc", "mean"), mod=("mod", "mean"))   # non-stimulated areas pooled per session
    stim_inc = st.set_index("site_idx").inc
    order_pooled = all(stim_inc.get(k, 0) > nsp.inc.get(k, 0) for k in stim_inc.index)
    order_region = all(stim_inc.get(k, 0) > v for k, v in zip(ns.site_idx, ns.inc))
    ratio_a = st[st.depth == "deep"]["mod"].mean() / st[st.depth == "superficial"]["mod"].mean()
    anyw = d[d.group.isin(claar.CORTEX) & d.stim & d.window.isin(["early", "off", "rebound"])].groupby(["site_idx", "depth", "unit"])["mod"].any().reset_index()
    anyw = anyw.groupby(["site_idx", "depth"])["mod"].mean().reset_index()
    ratio_b = anyw[anyw.depth == "deep"]["mod"].mean() / anyw[anyw.depth == "superficial"]["mod"].mean()
    rows.append({"level": c[0], "trials": c[1], "pre": c[2], "FDR": c[3], "stim sites": len(st),
                 "non-stim mean %": round(100 * ns["mod"].mean(), 2), "non-stim max %": round(100 * ns["mod"].max(), 2),
                 "non-stim pooled per session mean %": round(100 * nsp["mod"].mean(), 2), "max %": round(100 * nsp["mod"].max(), 2),
                 "stim mean %": round(100 * st["mod"].mean(), 2), "stim max %": round(100 * st["mod"].max(), 2), "stim min %": round(100 * st["mod"].min(), 2),
                 "inc stim > non-stim (pooled)": order_pooled, "(every region)": order_region,
                 "ratio deep/sup (a)": round(ratio_a, 2), "ratio (b)": round(ratio_b, 2)})
f4a = pd.DataFrame(rows)
f4a["values within tol"] = (abs(f4a["non-stim mean %"] - 0.5) <= 0.05) & (abs(f4a["non-stim max %"] - 2.6) <= 0.26) & (abs(f4a["stim mean %"] - 17.8) <= 1.78) & (abs(f4a["stim max %"] - 83.3) <= 8.33)
f4a["ratio 2-4"] = f4a["ratio deep/sup (a)"].between(2, 4) | f4a["ratio (b)"].between(2, 4)
print(f4a.to_string(index=False))
g = area_fractions(FL[("high", "awake", "adjacent", "site")], "early")
print("\nper site and area, high current, all awake, adjacent, per-site FDR (fractions):")
print(g.round(3).to_string(index=False))

**Result: not reproduced.** Modulated units at 2 to 25 ms are far more frequent than the paper reports in other cortex (mean 2.4 to 5.1% per region against 0.5%, maximum 29 to 50% against 2.6%) and in stimulated cortex (mean 24 to 39% against 17.8%), under every reading; the stimulated cortex does not exceed the other areas at every site. The deep over superficial ratio is between 2 and 4 in 15 of 16 readings. Pooling the non-stimulated areas of each session gives a mean of 2.1 to 4.3% and a maximum of 20%, also far above 0.5% and 2.6%.

## fig4be-off-period

> In the off period (25 to 150 ms) very few neurons increased their spiking (mean 0.4%, maximum 2.8%), spiking was
> significantly reduced in 18 of 22 subjects, and non-stimulated areas showed decreases only with deep stimulation
> (mean 10.1%, maximum 55.7%).

Readings of "spiking significantly reduced" in a site: (a) at least one stimulated-cortex RS unit decreased; (b) at
least one cortical RS unit decreased; (c) the summed spike count of the stimulated-cortex RS units lower after than
before by the signed-rank test.

In [ ]:
POP = {}
for lvl in LEVELS:
    for tr in AWAKE:
        for w in ("off", "rebound"):
            for pre in claar.PRE_READINGS:
                rows = []
                for r in res:
                    x = r["pop"]
                    if len(x):
                        x = x[(x.level == lvl) & (x.trials == tr) & (x.window == w) & (x.pre == pre)]
                        rows.append(x.assign(subject=r["subject"], site=r["site"], depth=r["depth"]))
                POP[(lvl, tr, w, pre)] = pd.concat(rows).reset_index(drop=True)
print("population tests:", {k: len(v) for k, v in list(POP.items())[:2]})

In [ ]:
rows = []
for c in combos:
    d = FL[c]
    g = area_fractions(d, "off")
    st = d[(d.window == "off") & d.group.isin(claar.CORTEX)]
    red_a = st[st.stim].groupby("site_idx").dec.any()
    red_b = st.groupby("site_idx").dec.any()
    pop = POP[(c[0], c[1], "off", c[2])]
    red_c = pop[pop.area == "stim"].pipe(lambda x: (x.p < 0.05) & (x.sign < 0))
    nsd = g[(~g.stim) & (g.depth == "deep")]
    nss = g[(~g.stim) & (g.depth == "superficial")]
    rows.append({"level": c[0], "trials": c[1], "pre": c[2], "FDR": c[3],
                 "inc mean %": round(100 * g.inc.mean(), 2), "inc max %": round(100 * g.inc.max(), 2),
                 "reduced (a)": f"{int(red_a.sum())}/{len(red_a)}", "(b)": f"{int(red_b.sum())}/{len(red_b)}", "(c)": f"{int(red_c.sum())}/{len(red_c)}",
                 "non-stim dec deep mean %": round(100 * nsd.dec.mean(), 2), "max %": round(100 * nsd.dec.max(), 2),
                 "sup sites with non-stim dec": int((nss.groupby("site_idx").dec.max() > 0).sum()), "of": nss.site_idx.nunique()})
f4b = pd.DataFrame(rows)
f4b["values within tol"] = (abs(f4b["inc mean %"] - 0.4) <= 0.04) & (abs(f4b["inc max %"] - 2.8) <= 0.28) & (abs(f4b["non-stim dec deep mean %"] - 10.1) <= 1.01) & (abs(f4b["max %"] - 55.7) <= 5.57)
print(f4b.to_string(index=False))
print("\nexpected: increased mean 0.4%, max 2.8%; reduced in 18 of 22 subjects; non-stim decreased with deep mean 10.1%, max 55.7%, none with superficial")

**Result: not reproduced.** Increases at 25 to 150 ms are few (mean 0.7 to 1.4%) but above the paper's 0.4% (maximum 11 to 19% against 2.8%); spiking is reduced in 17 to 22 of 22 stimulated areas (reading a) against 18 of 22; decreases in other cortex with deep stimulation average 16 to 28% against 10.1%; and 2 to 6 of 7 superficial sites also show decreases outside the stimulated area, where the paper reports none.

## fig4cf-rebound

> In the rebound window (150 to 300 ms), all 16 deep-stimulation subjects showed significant rebound excitation in
> stimulated cortex against 1 of 6 superficial subjects; 6 of 16 deep subjects showed it in non-stimulated areas, and
> none of 7 superficial subjects did.

In [ ]:
rows = []
for c in combos:
    d = FL[c]
    x = d[(d.window == "rebound") & d.group.isin(claar.CORTEX)]
    a_st = x[x.stim].groupby(["site_idx", "depth"]).inc.any().reset_index()
    a_ns = x[~x.stim].groupby(["site_idx", "depth"]).inc.any().reset_index()
    pop = POP[(c[0], c[1], "rebound", c[2])]
    pop["inc"] = (pop.p < 0.05) & (pop.sign > 0)
    out = {"level": c[0], "trials": c[1], "pre": c[2], "FDR": c[3]}
    for rd, st, ns in (("a", a_st, a_ns), ("b", pop[pop.area == "stim"], pop[pop.area == "nonstim"])):
        for dep in ("deep", "superficial"):
            out[f"{rd} stim {dep}"] = f"{int(st[st.depth == dep].inc.sum())}/{(st.depth == dep).sum()}"
            out[f"{rd} non-stim {dep}"] = f"{int(ns[ns.depth == dep].inc.sum())}/{(ns.depth == dep).sum()}"
        ds, ss = st[st.depth == "deep"].inc, st[st.depth == "superficial"].inc
        dn, sn = ns[ns.depth == "deep"].inc, ns[ns.depth == "superficial"].inc
        out[f"{rd} holds"] = bool(ds.mean() >= 0.9 and ss.mean() <= 1 / 6 and abs(dn.mean() - 6 / 16) <= 0.1 * 6 / 16 and sn.sum() == 0)
    rows.append(out)
f4c = pd.DataFrame(rows)
print(f4c.to_string(index=False))

**Result: partially reproduced.** In stimulated cortex the counts match at the medium current: 16 of 16 deep and 1 of 6 superficial sites with a significantly increased RS unit (the paper's 16/16 and 1/6, and its denominators). Outside the stimulated area, deep sites show rebound in 14 to 17 of 17 (reading a) or 5 to 10 of 17 (reading b) against 6 of 16, and 1 to 5 of 7 superficial sites show it against none. No reading meets every part of the criterion.

## fig4-supp1-fs-like-rs

> FS neurons behave like RS neurons in their modulation by stimulation depth and time window.

The analyses of fig4ad, fig4be and fig4cf on FS units, FDR over FS units; unit counts in fig4-unit-counts.

In [ ]:
rows = []
for c in combos:
    for cell in ("RS", "FS"):
        d = FL[c] if cell == "RS" else FL[c + ("FS",)]
        e = area_fractions(d, "early")
        o = area_fractions(d, "off")
        rb = d[(d.window == "rebound") & d.group.isin(claar.CORTEX) & d.stim].groupby(["site_idx", "depth"]).inc.any().reset_index()
        rows.append({"level": c[0], "trials": c[1], "pre": c[2], "FDR": c[3], "cell": cell,
                     "early inc stim %": round(100 * e[e.stim].inc.mean(), 1), "early inc non-stim %": round(100 * e[~e.stim].inc.mean(), 1),
                     "off dec %": round(100 * o.dec.mean(), 1), "off inc %": round(100 * o.inc.mean(), 1),
                     "rebound stim deep": f"{int(rb[rb.depth == 'deep'].inc.sum())}/{(rb.depth == 'deep').sum()}",
                     "rebound stim sup": f"{int(rb[rb.depth == 'superficial'].inc.sum())}/{(rb.depth == 'superficial').sum()}"})
fs = pd.DataFrame(rows)
fs["qualitative"] = (fs["early inc stim %"] > fs["early inc non-stim %"]) & (fs["off dec %"] > fs["off inc %"])
print(fs.to_string(index=False))

**Result: partially reproduced.** FS units follow the RS pattern under every reading (more increases in stimulated than other cortex at 2 to 25 ms, decreases dominating at 25 to 150 ms, rebound in 14 of 15 deep stimulated areas), but rebound also appears in 2 to 5 of 6 superficial sites, and the FS counts (68, 203, 149, 518) are far below the legend's (180, 585, 387, 1422).

## Figure 4 with the three currents pooled (a reading outside the claims file)

The Figure 4 legend's RS counts are three times the units in the files (fig4-unit-counts), which suggests that each unit was counted once per current. This reading is not in the claims file and is asked of a reader (questions/000458.yaml); it is shown here as a variant and does not decide an outcome. Each unit is tested at each current (all awake trials or quiet trials, both pre-stimulus windows, BH per site, window and current), and the fractions count unit x current.

In [ ]:
rows = []
for tr in AWAKE:
    for pre in claar.PRE_READINGS:
        for cell in ("RS", "FS"):
            parts = []
            for lvl in ("high", "medium", "low"):
                x = flags_all(lvl, tr, pre, "site", cell=cell)
                parts.append(x.assign(unit=x.unit.astype(str) + "-" + lvl))
            d = pd.concat(parts, ignore_index=True)
            e, o = area_fractions(d, "early"), area_fractions(d, "off")
            x = d[(d.window == "rebound") & d.group.isin(claar.CORTEX)]
            rb_st = x[x.stim].groupby(["site_idx", "depth"]).inc.any().reset_index()
            rb_ns = x[~x.stim].groupby(["site_idx", "depth"]).inc.any().reset_index()
            nsd, nss = o[(~o.stim) & (o.depth == "deep")], o[(~o.stim) & (o.depth == "superficial")]
            red = d[(d.window == "off") & d.group.isin(claar.CORTEX) & d.stim].groupby("site_idx").dec.any()
            ns_e = e[~e.stim]
            rows.append({"trials": tr, "pre": pre, "cell": cell,
                         "units x currents stim/non-stim": f"{int(e[e.stim].n.sum())}/{int(ns_e.n.sum())}",
                         "early: non-stim mean %": round(100 * ns_e["mod"].mean(), 2), "max %": round(100 * ns_e["mod"].max(), 2),
                         "stim mean %": round(100 * e[e.stim]["mod"].mean(), 2), "stim max %": round(100 * e[e.stim]["mod"].max(), 2),
                         "off: inc mean %": round(100 * o.inc.mean(), 2), "inc max %": round(100 * o.inc.max(), 2),
                         "reduced (a)": f"{int(red.sum())}/{len(red)}",
                         "non-stim dec deep mean %": round(100 * nsd.dec.mean(), 2), "dec max %": round(100 * nsd.dec.max(), 2),
                         "sup sites with non-stim dec": f"{int((nss.groupby('site_idx').dec.max() > 0).sum())}/{nss.site_idx.nunique()}",
                         "rebound stim deep": f"{int(rb_st[rb_st.depth == 'deep'].inc.sum())}/{(rb_st.depth == 'deep').sum()}",
                         "stim sup": f"{int(rb_st[rb_st.depth == 'superficial'].inc.sum())}/{(rb_st.depth == 'superficial').sum()}",
                         "non-stim deep": f"{int(rb_ns[rb_ns.depth == 'deep'].inc.sum())}/{(rb_ns.depth == 'deep').sum()}",
                         "non-stim sup": f"{int(rb_ns[rb_ns.depth == 'superficial'].inc.sum())}/{(rb_ns.depth == 'superficial').sum()}"})
f4p = pd.DataFrame(rows)
print(f4p.to_string(index=False))
print("\nexpected: early non-stim 0.5% (max 2.6%), stim 17.8% (max 83.3%); off inc 0.4% (max 2.8%), reduced 18 of 22, "
      "non-stim dec deep 10.1% (max 55.7%), none superficial; rebound stim 16/16 deep, 1/6 superficial, non-stim 6/16 deep, 0/7 superficial")

**Result: variant only.** With the three currents pooled, RS fractions stay far from the paper's: at 2 to 25 ms 2.6 to 3.2% of non-stimulated units (maximum 33%) and 26 to 29% of stimulated units are modulated, against 0.5% and 17.8%; at 25 to 150 ms increases average 0.7 to 0.8% (maximum 7.4%) and decreases outside the stimulated area with deep stimulation 17.2 to 17.3% (maximum 70 to 72%), against 0.4%, 2.8%, 10.1% and 55.7%, with 2 or 3 of 7 superficial sites showing such decreases; rebound outside the stimulated area appears in 15 or 16 of 17 deep and 2 of 7 superficial sites, against 6 of 16 and none. This reading does not change any Figure 4 outcome.

## fig2c-superficial-quiescence, fig2c-deep-triphasic-cortex, fig2f-thalamus-triphasic

Population rate: mean over units of the trial-averaged rate, 1 ms or 5 ms bins; baseline is the mean rate from
-2 to 0 s. Quiescence readings (a) below the baseline mean, (b) at zero, from 25 ms on.

In [ ]:
def pop_rate(r, level, trials, pop, bin_ms):
    t, rate = r["rate"][(level, trials)]
    u = r["units"]
    mask = (u.stim & u.rs).values if pop == "stimRS" else (u.group == "SM-TH").values
    if mask.sum() == 0:
        return None, None
    x = rate[mask].mean(0)
    if bin_ms > 1:
        x = x.reshape(-1, bin_ms).mean(1)
        t = t.reshape(-1, bin_ms).mean(1)
    return t, x

def triphasic(level, trials, pop, bin_ms, early, late, sites_filter):
    rows = []
    for k, r in enumerate(res):
        if not sites_filter(r):
            continue
        t, x = pop_rate(r, level, trials, pop, bin_ms)
        if x is None:
            continue
        base = x[t < 0].mean()
        sd = x[t < 0].std()
        e = x[(t >= early[0]) & (t <= early[1])].max()
        l = x[(t >= late[0]) & (t <= late[1])].max()
        rows.append({"subject": r["subject"], "site": r["site"], "base": base, "early peak": e, "late peak": l,
                     "quiescence a ms": 1000 * claar.quiescence(t, x, base), "quiescence b ms": 1000 * claar.quiescence(t, x, base, zero=True),
                     "quiescence a, gaps < 5 ms ignored": 1000 * claar.quiescence(t, x, base, max_gap=0.005),
                     "late z": (l - base) / sd if sd > 0 else np.inf,
                     "triphasic": bool(e > base and claar.quiescence(t, x, base) > 0 and l > base),
                     "triphasic, >= 10 ms below": bool(e > base and claar.quiescence(t, x, base, max_gap=0.005) >= 0.010 and l > base)})
    return pd.DataFrame(rows)

In [ ]:
rows = []
for lvl in LEVELS:
    for tr in AWAKE:
        for b in (1, 5):
            d = triphasic(lvl, tr, "stimRS", b, (0.002, 0.025), (0.15, 0.3), lambda r: r["site"] == "MOs-superficial")
            rows.append({"level": lvl, "trials": tr, "bin ms": b, "sites": len(d), "quiescence a": f"{d['quiescence a ms'].mean():.1f} +/- {stats.sem(d['quiescence a ms']):.1f}",
                         "a, gaps < 5 ms ignored": f"{d['quiescence a, gaps < 5 ms ignored'].mean():.1f}",
                         "quiescence b": f"{d['quiescence b ms'].mean():.1f} +/- {stats.sem(d['quiescence b ms']):.1f}",
                         "a within 94.2+/-16.1 or 44.3+/-16.4": bool(abs(d['quiescence a ms'].mean() - 94.2) <= 16.1 or abs(d['quiescence a ms'].mean() - 44.3) <= 16.4),
                         "b within": bool(abs(d['quiescence b ms'].mean() - 94.2) <= 16.1 or abs(d['quiescence b ms'].mean() - 44.3) <= 16.4)})
sq = pd.DataFrame(rows)
print(sq.to_string(index=False))
print("\nper site, high current, all awake, 1 ms bins:")
print(triphasic("high", "awake", "stimRS", 1, (0.002, 0.025), (0.15, 0.3), lambda r: r["site"] == "MOs-superficial").round(2).to_string(index=False))

**Result: reproduced.** In the two superficial MOs sites with units, MO RS population rates stay below baseline after 25 ms for 84.0 +/- 25.0 ms (high current, all awake, 1 ms bins), within 94.2 +/- 16.1; other readings give 81.5 to 115 ms below baseline, and 30 to 35 ms at zero rate, the latter within the Discussion's 44.3 +/- 16.4. Two sites are few. The measure is sensitive to single bins: when runs above baseline shorter than 5 ms do not end the period, the below-baseline quiescence lasts 110 to 121 ms, within 94.2 +/- 16.1 only with 5 ms bins and quiet trials at the high current (110.0 ms). The zero-rate reading (b) is within the Discussion's 44.3 +/- 16.4 in all eight readings, so the outcome does not rest on the single-bin reading alone.

In [ ]:
rows = []
for lvl in LEVELS:
    for tr in AWAKE:
        for b in (1, 5):
            for setname, flt in (("deep MOs", lambda r: r["site"] == "MOs-deep"), ("all deep", lambda r: r["depth"] == "deep")):
                d = triphasic(lvl, tr, "stimRS", b, (0.002, 0.025), (0.15, 0.3), flt)
                q = d["quiescence a ms"].mean(), d["quiescence b ms"].mean()
                rows.append({"level": lvl, "trials": tr, "bin": b, "set": setname, "sites": len(d),
                             "early peak Hz": f"{d['early peak'].mean():.1f} +/- {stats.sem(d['early peak']):.1f}",
                             "quiescence a ms": f"{q[0]:.1f} +/- {stats.sem(d['quiescence a ms']):.1f}", "quiescence b ms": f"{q[1]:.1f}",
                             "rebound peak Hz": f"{d['late peak'].mean():.1f} +/- {stats.sem(d['late peak']):.1f}",
                             "triphasic": f"{int(d.triphasic.sum())}/{len(d)}", ">= 10 ms below": f"{int(d['triphasic, >= 10 ms below'].sum())}/{len(d)}",
                             "quiescence a, gaps < 5 ms ignored": f"{d['quiescence a, gaps < 5 ms ignored'].mean():.1f}",
                             "all within": bool(abs(d['early peak'].mean() - 38.1) <= 4.2 and abs(d['late peak'].mean() - 7.9) <= 0.7 and
                                                any(abs(v - 125.0) <= 5.5 or abs(v - 127.8) <= 4.1 for v in q))})
dt = pd.DataFrame(rows)
print(dt.to_string(index=False))
print("\nper site, high current, all awake, 1 ms bins, all deep sites:")
print(triphasic("high", "awake", "stimRS", 1, (0.002, 0.025), (0.15, 0.3), lambda r: r["depth"] == "deep").round(2).to_string(index=False))

**Result: partially reproduced.** All 16 deep sites with stimulated-cortex RS units show the triphasic sequence. Quiescence below baseline lasts 108.7 to 131.9 ms (within 125.0 +/- 5.5 or 127.8 +/- 4.1 in several readings), but no reading puts the three values within tolerance: the early peak is 20 to 31 Hz in 5 ms bins and 44 to 65 Hz in 1 ms bins against 38.1 +/- 4.2, and the rebound peak 8.5 to 14.6 Hz against 7.9 +/- 0.7. Requiring at least 10 ms below baseline, with gaps under 5 ms ignored, keeps 16 of 16 sites triphasic.

In [ ]:
rows = []
for lvl in LEVELS:
    for tr in AWAKE:
        for b in (1, 5):
            d = triphasic(lvl, tr, "smth", b, (0.002, 0.025), (0.075, 0.3), lambda r: r["depth"] == "deep")
            s = triphasic(lvl, tr, "smth", b, (0.002, 0.025), (0.075, 0.3), lambda r: r["depth"] == "superficial")
            fl = FL[(lvl, tr, "adjacent", "site")]
            sup_inc = fl[(fl.window == "rebound") & (fl.group == "SM-TH") & (fl.depth == "superficial")].groupby("site_idx").inc.any()
            q = d["quiescence a ms"].mean(), d["quiescence b ms"].mean()
            rows.append({"level": lvl, "trials": tr, "bin": b, "deep sites": len(d),
                         "early peak": f"{d['early peak'].mean():.1f} +/- {stats.sem(d['early peak']):.1f}",
                         "suppression a ms": f"{q[0]:.1f} +/- {stats.sem(d['quiescence a ms']):.1f}", "b ms": f"{q[1]:.1f}",
                         "triphasic": f"{int(d.triphasic.sum())}/{len(d)}", ">= 10 ms below": f"{int(d['triphasic, >= 10 ms below'].sum())}/{len(d)}",
                         "rebound peak": f"{d['late peak'].mean():.1f} +/- {stats.sem(d['late peak']):.1f}",
                         "sup sites": len(s), "sup no rebound (a: z<=3)": f"{int((s['late z'] <= 3).sum())}/{len(s)}",
                         "sup no rebound (b: no unit inc)": f"{int((~sup_inc).sum())}/{len(sup_inc)}",
                         "values within": bool(abs(d['early peak'].mean() - 22.3) <= 3.2 and abs(d['late peak'].mean() - 14.7) <= 1.0 and any(abs(v - 74.6) <= 15.5 for v in q))})
th = pd.DataFrame(rows)
print(th.to_string(index=False))
print("\nper site, high current, all awake, 1 ms bins, SM-TH:")
print(triphasic("high", "awake", "smth", 1, (0.002, 0.025), (0.075, 0.3), lambda r: True).round(2).to_string(index=False))

**Result: partially reproduced.** SM-TH rates in the 14 deep sites with SM-TH units show early excitation, suppression and rebound, triphasic in 12 of 14 sites (per-site table, high current). The suppression (59.5 to 100.7 ms) is within 74.6 +/- 15.5 in most readings, but no reading puts the two peaks within tolerance together (early 13 to 32 Hz against 22.3 +/- 3.2; rebound 14.7 to 21.7 Hz against 14.7 +/- 1.0). With superficial stimulation 3 or 4 of 4 sites have no significantly increased SM-TH unit at 150 to 300 ms. Requiring at least 10 ms below baseline gives the same counts of triphasic sites in every reading.

## fig5a-thalamus-modulated

> SM-TH neurons were almost eight times more likely to be modulated by deep than by superficial stimulation
> (37.9 +/- 20.6% against 5.0 +/- 4.1%).

Readings: (a) one window 2 to 300 ms; (b) modulated in any of 2-25, 25-150, 150-300 ms.

In [ ]:
rows = []
for c in combos:
    d = FL[c]
    x = d[d.group == "SM-TH"]
    a = x[x.window == "2-300"].groupby(["site_idx", "depth"])["mod"].mean().reset_index()
    b = x[x.window.isin(["early", "off", "rebound"])].groupby(["site_idx", "depth", "unit"])["mod"].any().groupby(["site_idx", "depth"]).mean().reset_index()
    for rd, g in (("a", a), ("b", b)):
        s, dp = 100 * g[g.depth == "superficial"]["mod"], 100 * g[g.depth == "deep"]["mod"]
        tt = stats.ttest_ind(s, dp)
        rows.append({"level": c[0], "trials": c[1], "pre": c[2], "FDR": c[3], "reading": rd,
                     "superficial %": f"{s.mean():.1f} +/- {stats.sem(s):.1f} (n={len(s)})", "deep %": f"{dp.mean():.1f} +/- {stats.sem(dp):.1f} (n={len(dp)})",
                     "p": f"{tt.pvalue:.3g}", "holds": bool(abs(s.mean() - 5.0) <= 4.1 and abs(dp.mean() - 37.9) <= 20.6 and tt.pvalue < 0.05 and dp.mean() > s.mean())})
f5a = pd.DataFrame(rows)
print(f5a.to_string(index=False))

**Result: reproduced.** With one 2 to 300 ms window and pooled FDR, 1.9 +/- 1.5% of SM-TH units are modulated by superficial stimulation (n = 4 sites) and 42.7 +/- 5.8% by deep stimulation (n = 14), within 5.0 +/- 4.1 and 37.9 +/- 20.6, t-test p = 0.002. 15 of 32 readings meet the criterion; deep exceeds superficial with p < 0.05 in all 32.

## fig5c-burst-probability-depth

> Deep stimulation was significantly more likely than superficial stimulation to evoke SM-TH bursts 75 to 300 ms
> after the stimulus (median 0.12 [0.02 to 0.2] against 0 [0 to 0.02]).

Readings: (a) per site, the fraction of trials in which any SM-TH unit bursts; (b) the mean over SM-TH units of each
unit's fraction of trials with a burst.

In [ ]:
def burst_prob(r, level, trials):
    b = r["burst"][(level, trials)]
    m = (r["units"].group == "SM-TH").values
    if m.sum() == 0 or b.shape[1] == 0:
        return None
    b = b[m]
    return {"a": float(b.any(axis=0).mean()), "b": float(b.mean(axis=1).mean())}

def iqr(x):
    q = np.percentile(x, [25, 50, 75])
    return f"{q[1]:.3f} [{q[0]:.3f}-{q[2]:.3f}] (n={len(x)})"

rows = []
for lvl in LEVELS:
    for tr in AWAKE:
        d = pd.DataFrame([{"depth": r["depth"], **bp} for r in res if (bp := burst_prob(r, lvl, tr)) is not None])
        for rd in ("a", "b"):
            s, dp = d[d.depth == "superficial"][rd], d[d.depth == "deep"][rd]
            mw = stats.mannwhitneyu(dp, s, alternative="two-sided")
            rows.append({"level": lvl, "trials": tr, "reading": rd, "superficial": iqr(s), "deep": iqr(dp), "U p": f"{mw.pvalue:.3g}",
                         "holds": bool(abs(np.median(dp) - 0.12) <= 0.012 and np.median(s) == 0 and mw.pvalue < 0.05 and np.median(dp) > np.median(s))})
f5c = pd.DataFrame(rows)
print(f5c.to_string(index=False))

**Result: partially reproduced.** With reading (b), the mean over SM-TH units of the fraction of trials with a burst, deep sites have a median of 0.131 [0.035 to 0.293] at the high current (within 0.12 +/- 0.012) and the Mann-Whitney test gives p = 0.025, deep higher; the superficial median is 0.003 [0.002 to 0.029], which is 0 at the paper's printed precision but not exactly 0 as the criterion requires. Reading (a) gives far higher probabilities (0.87 deep).

## fig5e-early-latency and fig5f-rebound-latency

Responsive units: significantly increased in the window (stimulated-cortex RS; all SM-TH). Per unit, the median or
mean over trials of the first-spike latency; per population, the median over its units; per site.

In [ ]:
def latencies(c, lat_window, resp_window, summary, trials_state=None, depth=None, flags=None):
    d = FL[c] if flags is None else flags
    level, trials = c[0], c[1]
    rows = []
    for k, r in enumerate(res):
        if depth and r["depth"] != depth:
            continue
        st = trials_state or trials
        if (level, st, lat_window) not in r["latency"]:
            continue
        lat = r["latency"][(level, st, lat_window)]
        x = d[(d.site_idx == k) & (d.window == resp_window) & d.inc]
        u = r["units"].reset_index(drop=True)
        out = {"subject": r["subject"], "site": r["site"], "depth": r["depth"]}
        for pop, mask in (("cortex", (u.stim & u.rs)), ("SM-TH", (u.group == "SM-TH"))):
            ids = set(x.unit) & set(u[mask].unit)
            rowsel = u.index[u.unit.isin(ids)].values
            if len(rowsel) == 0:
                out[pop] = np.nan
                continue
            f = np.nanmedian if summary == "median" else np.nanmean
            per_unit = f(lat[rowsel], axis=1)
            out[pop] = 1000 * np.nanmedian(per_unit)
            out[pop + " units"] = len(rowsel)
        rows.append(out)
    return pd.DataFrame(rows)

In [ ]:
rows = []
for c in combos:
    for summ in ("median", "mean"):
        for dep in ("deep", None):
            d = latencies(c, "early", "early", summ, depth=dep).dropna(subset=["cortex", "SM-TH"])
            if len(d) < 2:
                continue
            tt = stats.ttest_rel(d["SM-TH"], d["cortex"])
            diff = (d["SM-TH"] - d["cortex"]).mean()
            rows.append({"level": c[0], "trials": c[1], "pre": c[2], "FDR": c[3], "unit summary": summ, "sites": dep or "all", "n": len(d),
                         "cortex ms": f"{d.cortex.mean():.1f} +/- {stats.sem(d.cortex):.1f}", "SM-TH ms": f"{d['SM-TH'].mean():.1f} +/- {stats.sem(d['SM-TH']):.1f}",
                         "diff ms": round(diff, 2), "p": f"{tt.pvalue:.3g}",
                         "holds": bool(abs(d.cortex.mean() - 4.4) <= 0.9 and abs(d['SM-TH'].mean() - 11.4) <= 2.8 and (abs(diff - 6.9) <= 0.69 or abs(diff - 6.7) <= 0.67) and tt.pvalue < 0.05 and diff > 0)})
f5e = pd.DataFrame(rows)
print(f5e.to_string(index=False))

**Result: partially reproduced.** Stimulated-cortex RS units fire first, 4.0 to 5.2 ms (within 4.4 +/- 0.9), and SM-TH units later, 7.2 to 9.8 ms (within 11.4 +/- 2.8 in some readings), p < 0.05 in all high-current readings; but the mean difference is 3.2 to 4.9 ms against 6.9 (or 6.7), out of tolerance in every reading. 6 to 12 sites have responsive units in both populations, the paper 13.

In [ ]:
rows = []
for c in combos:
    for summ in ("median", "mean"):
        for lw in ("late75", "late100"):
            for resp in ("rebound", lw):
                d = latencies(c, lw, resp, summ, depth="deep").dropna(subset=["cortex", "SM-TH"])
                if len(d) < 2:
                    continue
                tt = stats.ttest_rel(d["cortex"], d["SM-TH"])
                diff = (d["cortex"] - d["SM-TH"]).mean()
                rows.append({"level": c[0], "trials": c[1], "pre": c[2], "FDR": c[3], "summary": summ, "window": lw, "responsive in": resp, "n": len(d),
                             "SM-TH ms": f"{d['SM-TH'].mean():.1f} +/- {stats.sem(d['SM-TH']):.1f}", "cortex ms": f"{d.cortex.mean():.1f} +/- {stats.sem(d.cortex):.1f}",
                             "diff ms": round(diff, 1), "p": f"{tt.pvalue:.3g}",
                             "holds": bool(abs(d['SM-TH'].mean() - 148.2) <= 21.2 and abs(d.cortex.mean() - 184.2) <= 17.3 and (abs(diff - 36.0) <= 3.6 or abs(diff - 37.0) <= 3.7) and tt.pvalue < 0.05 and diff > 0)})
f5f = pd.DataFrame(rows)
print(f5f.to_string(index=False))
print("\ncombinations where the SM-TH fires first with p < 0.05:", int(((f5f["diff ms"] > 0) & (f5f.p.astype(float) < 0.05)).sum()), "of", len(f5f))

**Result: not reproduced.** In the late window the mean SM-TH latency (155.5 to 186.2 ms) is earlier than the cortical one (174.2 to 190.4 ms) in all 128 combinations of readings, but by 0.9 to 19.5 ms against 36.0, and significantly (p < 0.05) in 3 of 128.

## fig6b, fig6c, fig6-supp2: trial-by-trial correlations in deep sites

Per deep site with SM-TH and stimulated-cortex RS units, awake trials: the fraction of SM-TH units with a burst in
75-300 ms, the peak stimulated-cortex population rate in 75-300 ms (5 or 10 ms bins), the peak SM-TH population
rate, and the single-trial late ERP amplitude ((a) peak GFP, (b) peak |V| at the electrode with the largest
trial-averaged late component). Pearson r and p per site.

In [ ]:
t0 = time.time()
deep_sites = [(k, r) for k, r in enumerate(res) if r["depth"] == "deep" and (r["units"].group == "SM-TH").any() and (r["units"].stim & r["units"].rs).any()]
LATE = {}
for k, r in deep_sites:
    m = [mm for mm in metas if mm["subject"] == r["subject"] and mm["session"] == r["session"]][0]
    LATE[k] = claar.late_amplitudes(m, r["site"])
print(f"single-trial EEG for {len(deep_sites)} deep sites in {time.time() - t0:.0f} s")

def trial_table(k, r, level):
    ids = r["trial_ids"][(level, "awake")]
    b = r["burst"][(level, "awake")][(r["units"].group == "SM-TH").values]
    d = pd.DataFrame({"trial": ids, "burst_frac": b.mean(axis=0)})
    for bs in (0.005, 0.010):
        d[f"ctx_{int(bs * 1000)}"] = r["rate"][(level, "single", bs, "stimRS")]
        d[f"th_{int(bs * 1000)}"] = r["rate"][(level, "single", bs, "smth")]
    return d.merge(LATE[k][level], on="trial")

def corr_rows(xcol, ycol, level):
    out = []
    for k, r in deep_sites:
        d = trial_table(k, r, level)
        if d[xcol].std() == 0 or d[ycol].std() == 0:
            out.append({"subject": r["subject"], "site": r["site"], "r": np.nan, "p": np.nan, "n": len(d)})
            continue
        pr = stats.pearsonr(d[xcol], d[ycol])
        out.append({"subject": r["subject"], "site": r["site"], "r": pr.statistic, "p": pr.pvalue, "n": len(d)})
    return pd.DataFrame(out)

def summarize(pairs, exp_n, exp_r, tol_r):
    rows = []
    for lvl in LEVELS:
        for name, (x, y) in pairs.items():
            d = corr_rows(x, y, lvl)
            n_sig = int((d.p < 0.05).sum())
            rows.append({"level": lvl, "pair": name, "sites": len(d), "significant": f"{n_sig}/{len(d)}",
                         "mean r": f"{d.r.mean():.3f} +/- {stats.sem(d.r, nan_policy='omit'):.3f}",
                         "holds": bool(abs(d.r.mean() - exp_r) <= tol_r and abs(n_sig / len(d) - exp_n[0] / exp_n[1]) <= 0.1 * exp_n[0] / exp_n[1])})
    return pd.DataFrame(rows)

In [ ]:
f6b = summarize({"bursts vs cortex 5 ms": ("burst_frac", "ctx_5"), "bursts vs cortex 10 ms": ("burst_frac", "ctx_10")}, (8, 13), 0.4, 0.04)
print(f6b.to_string(index=False))
print("\nper site, high current, 5 ms bins:")
print(corr_rows("burst_frac", "ctx_5", "high").round(3).to_string(index=False))

**Result: not reproduced.** In the 13 deep sites with SM-TH and stimulated-cortex units (the paper's N), the trial-by-trial Pearson r between the SM-TH burst fraction and the peak cortical rate averages 0.13 to 0.24 against 0.4 +/- 0.04, with 6 to 10 sites significant against 8.

In [ ]:
f6c = summarize({"cortex 5 ms vs late GFP (a)": ("ctx_5", "late_gfp"), "cortex 10 ms vs late GFP (a)": ("ctx_10", "late_gfp"),
                 "cortex 5 ms vs electrode (b)": ("ctx_5", "late_electrode"), "cortex 10 ms vs electrode (b)": ("ctx_10", "late_electrode")}, (9, 13), 0.5, 0.04)
print(f6c.to_string(index=False))

**Result: not reproduced.** The peak cortical rate correlates with the single-trial late ERP amplitude with mean r 0.21 to 0.32 against 0.5 +/- 0.04; 7 to 11 of 13 sites are significant against 9.

In [ ]:
rows = []
for lvl in LEVELS:
    for amp in ("late_gfp", "late_electrode"):
        b = corr_rows("burst_frac", amp, lvl)
        for bs in ("th_5", "th_10"):
            t = corr_rows(bs, amp, lvl)
            nb, nt = int((b.p < 0.05).sum()), int((t.p < 0.05).sum())
            rows.append({"level": lvl, "late ERP": amp, "SM-TH rate bins": bs, "sites": len(b),
                         "bursts: significant": f"{nb}/{len(b)}", "bursts: mean r": f"{b.r.mean():.3f} +/- {stats.sem(b.r, nan_policy='omit'):.3f}",
                         "rate: significant": f"{nt}/{len(t)}", "rate: mean r": f"{t.r.mean():.3f}",
                         "holds": bool(abs(b.r.mean() - 0.5) <= 0.07 and nb >= len(b) - 1 and t.r.mean() < b.r.mean() and nt < nb)})
f6s = pd.DataFrame(rows)
print(f6s.to_string(index=False))

**Result: reproduced.** The late ERP amplitude (peak GFP, 75 to 300 ms) correlates with the SM-TH burst fraction in 13 of 13 sites at both currents, mean r 0.462 +/- 0.039 (high) and 0.455 (medium), within 0.5 +/- 0.07; its correlation with the peak SM-TH rate is lower (mean r 0.06 to 0.10, 4 of 13 significant). The electrode reading of the late ERP (b) does not meet the criterion: 12 of 13 sites significant at the high current (mean r 0.424) and 11 of 13 at the medium current (0.502).


## Figure 7: states

Per site, quiet, running and isoflurane trials of the intensity used; per animal (sites of one animal averaged);
session sets (a) deep sites, (b) all sites with units.

In [ ]:
def baseline_frame(level, window):
    rows = []
    for r in res:
        u = r["units"].reset_index(drop=True)
        for st in ("quiet", "running", "isoflurane"):
            if r["n"][(level, st)] == 0:
                continue
            b = r["baseline"][(level, st)][window]
            for pop, mask in (("cortex", (u.stim & u.rs).values), ("SM-TH", (u.group == "SM-TH").values)):
                if mask.sum():
                    rows.append({"subject": r["subject"], "depth": r["depth"], "state": st, "pop": pop, "rate": b[mask].mean()})
    return pd.DataFrame(rows)

exp7e = {("cortex", "quiet"): [(3.5, 0.2), (3.3, 0.2)], ("cortex", "running"): [(5.2, 0.5), (5.0, 0.5)], ("cortex", "isoflurane"): [(1.1, 0.2), (1.1, 0.2)],
         ("SM-TH", "quiet"): [(9.0, 1.4), (9.4, 1.5)], ("SM-TH", "running"): [(16.4, 1.5), (16.8, 1.6)], ("SM-TH", "isoflurane"): [(0.8, 0.4), (0.8, 0.4)]}
rows = []
for lvl in LEVELS:
    for win in ("-2-0", "-0.5-0"):
        d = baseline_frame(lvl, win)
        for setname, sub in (("a: deep", d[d.depth == "deep"]), ("b: all", d)):
            out = {"level": lvl, "window": win, "set": setname}
            within = [True, True]
            order = True
            for pop in ("cortex", "SM-TH"):
                w = sub[sub["pop"] == pop].pivot_table(index="subject", columns="state", values="rate", aggfunc="mean").dropna()
                for st in ("quiet", "running", "isoflurane"):
                    out[f"{pop} {st}"] = f"{w[st].mean():.2f} +/- {stats.sem(w[st]):.2f}"
                    for j in range(2):
                        within[j] &= abs(w[st].mean() - exp7e[(pop, st)][j][0]) <= exp7e[(pop, st)][j][1]
                order &= bool(w.running.mean() > w.quiet.mean() > w.isoflurane.mean())
                if pop == "cortex":
                    long = w.reset_index().melt(id_vars="subject", var_name="state", value_name="rate")
                    an = AnovaRM(long, "rate", "subject", within=["state"]).fit().anova_table
                    out["cortex RM ANOVA"] = f"F[{int(an['Num DF'].iloc[0])},{int(an['Den DF'].iloc[0])}]={an['F Value'].iloc[0]:.1f}, p={an['Pr > F'].iloc[0]:.2g}"
                    p1 = an["Pr > F"].iloc[0]
                else:
                    fr = stats.friedmanchisquare(w.quiet, w.running, w.isoflurane)
                    out["SM-TH Friedman"] = f"Q={fr.statistic:.1f}, p={fr.pvalue:.2g}, n={len(w)}"
                    p2 = fr.pvalue
            out["within (Results / Discussion)"] = f"{within[0]} / {within[1]}"
            out["order"] = order
            out["holds"] = bool(any(within) and order and p1 < 0.05 and p2 < 0.05)
            rows.append(out)
f7e = pd.DataFrame(rows)
print(f7e.to_string(index=False))

**Result: reproduced.** In the deep sites, high current, -2 to 0 s: stimulated-cortex RS 3.56, 4.97 and 1.02 Hz and SM-TH 9.11, 15.52 and 0.81 Hz (quiet, running, isoflurane), all within tolerance of the Results' values; RM ANOVA F[2,28] = 86.7, p = 1e-12 (15 animals); Friedman Q = 24.2, p = 5.7e-06 (13 animals), the paper's Q and p. Over all sites the cortex has the paper's 16 animals (F[2,30] = 80.7) and the means are within tolerance of the Results' or the Discussion's values.

In [ ]:
def lat_state(c, lat_window, resp_window, summ, depth, resp_from="state"):
    # responsive units from the trials of each state, or from all awake trials (resp_from="awake")
    rows = []
    for st in ("quiet", "running", "isoflurane"):
        cc = (c[0], st, c[2], c[3]) if st != "running" else (c[0], "running", "adjacent", c[3])
        if cc not in FL:
            FL[cc] = flags_all(*cc)
        flags = FL[(c[0], "awake", c[2], c[3])] if resp_from == "awake" else None
        d = latencies(cc, lat_window, resp_window, summ, depth=depth, flags=flags)
        d["state"] = st
        rows.append(d)
    d = pd.concat(rows)
    d = d.groupby(["subject", "state"])[["cortex", "SM-TH"]].mean().reset_index()
    full = d.dropna().groupby("subject").filter(lambda g: len(g) == 3)
    return full

def two_way(full):
    long = full.melt(id_vars=["subject", "state"], value_vars=["cortex", "SM-TH"], var_name="region", value_name="lat")
    return AnovaRM(long, "lat", "subject", within=["state", "region"]).fit().anova_table

rows = []
for lvl in LEVELS:
    for pre in claar.PRE_READINGS:
        for fam in ("site", "pooled"):
            for summ, rf in (("median", "state"), ("mean", "state"), ("median", "awake"), ("mean", "awake")):
                full = lat_state((lvl, None, pre, fam), "early", "early", summ, "deep", rf)
                if full.subject.nunique() < 3:
                    continue
                an = two_way(full)
                g = full.groupby("state")[["cortex", "SM-TH"]].mean()
                rows.append({"level": lvl, "pre": pre, "FDR": fam, "summary": summ, "responsive from": rf, "animals": full.subject.nunique(),
                             **{f"{p} {s}": round(g.loc[s, p], 2) for p in ("cortex", "SM-TH") for s in ("quiet", "running", "isoflurane")},
                             "state F, p": f"{an.loc['state', 'F Value']:.2f}, {an.loc['state', 'Pr > F']:.3g}",
                             "region F, p": f"{an.loc['region', 'F Value']:.1f}, {an.loc['region', 'Pr > F']:.3g}",
                             "holds": bool(an.loc["state", "Pr > F"] >= 0.05 and an.loc["region", "Pr > F"] < 0.05)})
f7f = pd.DataFrame(rows)
print(f7f.to_string(index=False))
if len(f7f):
    print("\nrows where the state effect is absent (p >= 0.05) and the region effect present:", int(f7f.holds.sum()), "of", len(f7f))
full = lat_state(("high", None, "adjacent", "site"), "early", "early", "median", "deep", "awake")
print("\nbounds on the state effect, high current, adjacent, per-site FDR, median, responsive from all awake trials "
      f"({full.subject.nunique()} animals): paired difference of state means, ms, mean [95% CI]")
w = full.pivot_table(index="subject", columns="state", values=["cortex", "SM-TH"])
for pop in ("cortex", "SM-TH"):
    for a, b in (("quiet", "running"), ("quiet", "isoflurane"), ("running", "isoflurane")):
        dd = (w[(pop, a)] - w[(pop, b)]).values
        ci = stats.t.interval(0.95, len(dd) - 1, loc=dd.mean(), scale=stats.sem(dd))
        print(f"  {pop} {a} - {b}: {dd.mean():.2f} [{ci[0]:.2f}, {ci[1]:.2f}]")

**Result: reproduced, resting on an assumption.** With responsive units taken from all awake trials, the two-way RM ANOVA finds no state effect (p = 0.08 to 0.90) and a region effect (p = 0.00016 to 0.017) in all 16 such readings (8 animals at the high current, 6 at the medium). With responsive units taken from each state's own trials only 3 animals have units in every state and cell, and no region effect is found (p about 0.3). The SM-TH latencies (7.1 to 11.6 ms) are mostly below 11.5, 11.3 and 10.9 +/- 0.8. The absence of a state effect rests on 6 to 8 animals against the paper's 13; at the high current (8 animals) the 95% confidence intervals of the paired state differences are within -1.3 to +0.6 ms in cortex and -7.5 to +2.4 ms in SM-TH, and the SM-TH quiet against running difference (0.92 ms [0.04, 1.80]) excludes zero without a correction for the six pairs.

In [ ]:
exp7g = {("cortex", "quiet"): (186.7, 4.6), ("cortex", "running"): (163.8, 5.0), ("cortex", "isoflurane"): (200.2, 7.5),
         ("SM-TH", "quiet"): (156.0, 7.1), ("SM-TH", "running"): (123.1, 3.9), ("SM-TH", "isoflurane"): (195.1, 9.5)}
rows = []
for lvl in LEVELS:
    for pre in claar.PRE_READINGS:
        for fam in ("site", "pooled"):
            for summ in ("median", "mean"):
                for lw in ("late100", "late75"):
                    for resp, rf in (("rebound", "state"), (lw, "state"), ("rebound", "awake"), (lw, "awake")):
                        full = lat_state((lvl, None, pre, fam), lw, resp, summ, "deep", rf)
                        if full.subject.nunique() < 3:
                            continue
                        an = two_way(full)
                        g = full.groupby("state")[["cortex", "SM-TH"]].mean()
                        pairs = [(p, a, b) for p in ("cortex", "SM-TH") for a, b in (("quiet", "running"), ("quiet", "isoflurane"), ("running", "isoflurane"))]
                        pv = []
                        for p, a, b in pairs:
                            w = full.pivot(index="subject", columns="state", values=p)
                            pv.append(stats.ttest_rel(w[a], w[b]).pvalue)
                        q = claar.bh(pv)
                        run_earlier = all(g.loc["running", p] < g.loc["quiet", p] for p in ("cortex", "SM-TH"))
                        within = all(abs(g.loc[s, p] - v[0]) <= v[1] for (p, s), v in exp7g.items())
                        rows.append({"level": lvl, "pre": pre, "FDR": fam, "summary": summ, "window": lw, "responsive in": resp, "from": rf, "animals": full.subject.nunique(),
                                     **{f"{p[:3]} {s[:4]}": round(g.loc[s, p], 1) for p in ("cortex", "SM-TH") for s in ("quiet", "running", "isoflurane")},
                                     "state p": f"{an.loc['state', 'Pr > F']:.3g}", "q ctx q-r": round(q[0], 3), "q th q-r": round(q[3], 3),
                                     "within": within,
                                     "state & run<quiet sig": bool(an.loc["state", "Pr > F"] < 0.05 and run_earlier and q[0] < 0.05 and q[3] < 0.05)})
f7g = pd.DataFrame(rows)
print(f7g.to_string(index=False))
if len(f7g):
    print("\nrows where state p < 0.05 and running is earlier than quiet in both regions by the corrected test:", int(f7g["state & run<quiet sig"].sum()), "of", len(f7g))
    print("rows with all six means within tolerance:", int(f7g.within.sum()))

**Result: partially reproduced, resting on the same assumption.** In all 64 readings the state effect is significant (p from 3e-08 to 3e-05) and running is earlier than quiet wakefulness in both regions by the corrected paired test (q <= 0.003), but no reading puts all six means within tolerance: the SM-TH latencies in quiet wakefulness (167 to 187 ms) are later than 156.0 +/- 7.1, and the isoflurane latencies (cortex 203 to 220 ms, SM-TH 198 to 221 ms) mostly later than 200.2 +/- 7.5 and 195.1 +/- 9.5.

In [ ]:
rows = []
for lvl in LEVELS:
    for rd in ("a", "b"):
        d = []
        for r in res:
            for st in ("quiet", "running", "isoflurane"):
                if r["n"][(lvl, st)] == 0:
                    continue
                bp = burst_prob(r, lvl, st)
                if bp is not None:
                    d.append({"subject": r["subject"], "depth": r["depth"], "state": st, "p": bp[rd]})
        d = pd.DataFrame(d)
        for setname, sub in (("a: deep", d[d.depth == "deep"]), ("b: all", d)):
            w = sub.pivot_table(index="subject", columns="state", values="p", aggfunc="mean").dropna()
            fr = stats.friedmanchisquare(w.quiet, w.running, w.isoflurane)
            med = w.median()
            rows.append({"level": lvl, "reading": rd, "set": setname, "animals": len(w),
                         **{s: iqr(w[s]) for s in ("quiet", "running", "isoflurane")}, "Friedman": f"Q={fr.statistic:.1f}, p={fr.pvalue:.3g}",
                         "holds": bool((abs(med.quiet - 0.1) <= 0.01 or abs(med.quiet - 0.095) <= 0.0095) and med.running == 0 and med.isoflurane == 0 and fr.pvalue < 0.05 and med.quiet >= med.max())})
f7h = pd.DataFrame(rows)
print(f7h.to_string(index=False))

**Result: partially reproduced.** Burst probability falls from quiet wakefulness to running to isoflurane (13 animals, Friedman p from 0.0002 to 0.0007, quiet highest in every reading), but the medians do not match: reading (b) gives 0.177, 0.121 and 0.005 at the high current and 0.150, 0.028 and 0.006 at the medium current against 0.1, 0 and 0.

## Summary

| Claim | Outcome |
|---|---|
| fig2c-superficial-quiescence | reproduced |
| fig2c-deep-triphasic-cortex | partially reproduced |
| fig2f-thalamus-triphasic | partially reproduced |
| fig3abc-decay-with-distance, fig3d-magnitudes | fig3abc partially reproduced, fig3d blocked: too large (`claar_lfp.ipynb`); the spiking part integrated as |z| comes near the paper's values |
| fig4-unit-counts | not reproduced (one third of the legend's counts; units x currents match) |
| fig4ad-initial-excitation | not reproduced |
| fig4be-off-period | not reproduced |
| fig4cf-rebound | partially reproduced |
| fig4-supp1-fs-like-rs | partially reproduced |
| fig5a-thalamus-modulated | reproduced |
| fig5c-burst-probability-depth | partially reproduced |
| fig5e-early-latency | partially reproduced |
| fig5f-rebound-latency | not reproduced |
| fig6a-csd-sink, fig6-supp1-erp-deep-csd | blocked: metadata insufficient (no cortical layers) |
| fig6b-bursts-cortical-rate | not reproduced |
| fig6c-cortical-rate-late-erp | not reproduced |
| fig6-supp2-late-erp-bursts | reproduced |
| fig7e-baseline-rates | reproduced |
| fig7f-early-latency-state | reproduced (assumption) |
| fig7g-rebound-latency-state | partially reproduced (assumption) |
| fig7h-burst-probability-state | partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's: the current analysed is not given and the highest and middle currents
are both run; units are those in the files (their quality threshold is not stated, and presence ratios go down to
0.02); stimulated cortex is the area group of the target (MO or SS) on any probe, as the files give no distance
to the stimulating electrode; the pre-stimulus window and the family of the Benjamini-Hochberg correction are not
given and both readings of each are run; for the state latencies of Figure 7F and 7G, responsive units are taken
from all awake trials, an assumption, because per-state selection leaves 3 animals. Variants outside the claims
file are shown and decide no outcome: Figure 4 with the three currents pooled (counting units x currents gives the legend's
first three counts exactly and 9486 against 9363), and the spiking part of Figure 3, with the signed z of the Methods and with |z|.

What was found about the dataset: units carry CCF areas without cortical layers, so layer 1 can only be approximated
(`claar_lfp.ipynb`) and the layer-resolved CSD analyses of Figure 6 cannot be run from the files; `waveform_duration` is in ms while its description says s; the RS unit
counts are one third of the Figure 4 legend's, and counting each unit once per current gives the legend's first three counts exactly and 9486
against 9363; one superficial site has no valid trial, which brings the dandiset's
8 superficial sites with units to the paper's 7.

Runtime of this notebook:

In [ ]:
print(f"{(time.time() - started) / 60:.1f} min")